# 1. Commits

A commit is a snapshot of the whole project, plus a message saying why. By
the end of this chapter the history looks like this:

```mermaid
gitGraph
    commit id: "volume + displacement"
    commit id: "block coefficient"
    commit id: "checks"
    commit id: "buoyancy"
    commit id: "ignore caches"
```

In [ ]:
!rm -rf ~/git-course/01-commits
!mkdir -p ~/git-course/01-commits
%cd ~/git-course/01-commits

## A repository

`git init` turns the current folder into a repository. It creates one
hidden folder, `.git`, where the history is kept.

In [ ]:
!git init

In [ ]:
!ls -a

In [ ]:
!git status

No commits yet, nothing to commit. Let's write some code.

In [ ]:
%%writefile hydrostatics.py
"""Hydrostatics for a box-shaped barge."""

RHO = 1025.0  # kg/m^3, sea water


def volume(length, beam, draft):
    """Immersed volume of the hull, in m^3."""
    return length * beam * draft


def displacement(length, beam, draft):
    """Mass of the water the hull pushes aside, in kg."""
    return RHO * volume(length, beam, draft)

In [ ]:
!git status

The file is **untracked**: git sees it but is not saving it. Git never
saves anything until you tell it to. First you **add** a file to the
staging area — the list of things that go into the next commit:

In [ ]:
!git add hydrostatics.py

In [ ]:
!git status

Now it is staged. **Commit** saves the snapshot:

In [ ]:
!git commit -m "Add volume and displacement for a box barge"

In [ ]:
!git log

## Changing the file

Real hulls are not boxes. The **block coefficient** `cb` is the fraction
of the box the hull really fills: 0.85 for a tanker, 0.5 for a patrol boat.

In [ ]:
%%writefile hydrostatics.py
"""Hydrostatics for a box-shaped barge."""

RHO = 1025.0  # kg/m^3, sea water


def volume(length, beam, draft, cb=1.0):
    """Immersed volume of the hull, in m^3."""
    return length * beam * draft * cb


def displacement(length, beam, draft, cb=1.0):
    """Mass of the water the hull pushes aside, in kg."""
    return RHO * volume(length, beam, draft, cb)

`git diff` shows what changed since the last commit. Lines starting with
`-` were removed, lines starting with `+` were added.

In [ ]:
!git diff

In [ ]:
!git add hydrostatics.py
!git commit -m "Scale volume by the block coefficient"

A good message says **why**, in a short line: "Add …", "Fix …", "Scale …".
Messages like `update` or `stuff` are no help to anyone later — including
you.

## A third commit

In [ ]:
%%writefile test_hydrostatics.py
from hydrostatics import displacement, volume

assert volume(10, 4, 2) == 80
assert volume(10, 4, 2, 0.5) == 40
assert displacement(10, 4, 2) == 82000  # 80 m^3 of sea water

print("all checks passed")

In [ ]:
!python test_hydrostatics.py

In [ ]:
!git add test_hydrostatics.py
!git commit -m "Add checks for volume and displacement"

In [ ]:
!git log --oneline

`--oneline` is the short view you will use every day. **HEAD** marks the
commit you are on now.

## Fixing the last commit

Buoyancy is the force the water pushes back with. Let's add it, and commit
it with a bad message on purpose.

In [ ]:
%%writefile -a hydrostatics.py


def buoyancy(length, beam, draft, cb=1.0):
    """Upward force on the hull, in N."""
    return 9.81 * displacement(length, beam, draft, cb)

In [ ]:
!git add hydrostatics.py
!git commit -m "stuff"

`--amend` replaces the last commit with a corrected one:

In [ ]:
!git commit --amend -m "Add buoyancy force from displacement"

In [ ]:
!git log --oneline

Only amend commits you have not shared yet (chapter 5 explains why).

`git show` prints one commit and what it changed:

In [ ]:
!git show HEAD

## Throwing away a change

Someone adds nonsense to the file:

In [ ]:
%%writefile -a hydrostatics.py

RHO = 1.2  # air?!

In [ ]:
# This one fails on purpose: the checks catch the mistake.
!python test_hydrostatics.py

`git restore` puts the file back to how it was in the last commit:

In [ ]:
!git restore hydrostatics.py

In [ ]:
!python test_hydrostatics.py

## Un-staging a file

A file with a password must never be committed. If you add it by mistake,
`git restore --staged` takes it back out of the staging area:

In [ ]:
%%writefile credentials.txt
secret-password-do-not-commit

In [ ]:
!git add credentials.txt
!git status

In [ ]:
!git restore --staged credentials.txt
!git status

## .gitignore

A `.gitignore` file lists files git should never track — secrets, and the
`__pycache__` folder Python creates.

In [ ]:
%%writefile .gitignore
__pycache__/
credentials.txt

In [ ]:
!git status

In [ ]:
!git add .gitignore
!git commit -m "Ignore caches and local credentials"

In [ ]:
!git log --oneline

## Summary

| I want to… | Command |
| - | - |
| start a repository | `git init` |
| see what is going on | `git status` |
| stage a file | `git add <file>` |
| save a snapshot | `git commit -m "message"` |
| see what changed | `git diff` |
| see the history | `git log --oneline` |
| fix the last commit | `git commit --amend -m "message"` |
| throw away a change | `git restore <file>` |
| un-stage a file | `git restore --staged <file>` |

## Exercises

1. Add a `waterplane_area(length, beam)` function and commit it with a good
   message.
2. Try `git log --stat` and `git log -p`. What do they add?

Next: [2. Branches](02-branches.ipynb)